In [1]:
from nd2reader import ND2Reader
import tifffile
import numpy as np
from matplotlib import pyplot as plt
import glob
import sys
import numpy as np
import os
import importlib
from scipy.ndimage import label
import natsort
import shutil
import re
import cv2
from skimage import restoration

This notebook handles the processing of our imaging datasets into a consistent format for training.

Currently, it can process a hand-corrected movie into a set of frames and masks and it can process a set of tiff files from Nikon Elements into a similar set of frames and masks.

It makes the training set such that each input image consists of two channels: Brightfield and Cy5 (Chlorophyll). The masks have a background of 0 and cells labeled with unique integers.

Images that have not been hand-corrected (no _seg.npy) are excluded from the training set (so they can be used for testing).


# Settings

In [2]:
train_path = '/Users/bebr1814/projects/anabaena/scratch_data/bulk_training_set'
test_path = '/Users/bebr1814/projects/anabaena/scratch_data/bulk_test_set'

In [15]:
def apply_gaussian_blur(img, kernel_size=5, sigma=1):
    """
    Apply Gaussian blur to an image.
    
    Parameters:
        img (numpy.ndarray): Input image (grayscale or color).
        kernel_size (int): Size of the Gaussian kernel (must be odd).
        sigma (float): Standard deviation of the Gaussian kernel.
    
    Returns:
        numpy.ndarray: Blurred image.
    """
    # Ensure kernel size is odd
    if kernel_size % 2 == 0:
        kernel_size += 1
    blurred = cv2.GaussianBlur(img, (kernel_size, kernel_size), sigma)
    return blurred

def normalize(img):
	img = img.astype(np.float32)
	img = apply_gaussian_blur(img, kernel_size=5, sigma=1)
	img = (img - img.min()) / (img.max() - img.min())
	img = (img * 255).astype(np.uint8)
	return img

In [17]:
# # test normalization
# img = tifffile.imread('/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena/set1.011/tif/set1.011_0002.brightfield.tiff')[0]
# plt.imshow(img, cmap='gray')
# plt.show()
# img = normalize(img)
# plt.imshow(img, cmap='gray')
# plt.show()

# Movies

In [4]:
# Exclude frames from training
test_frames = [15,30,45,60,75]

In [5]:
movie_mask = tifffile.imread('/Users/bebr1814/projects/anabaena/scratch_data/movies/2020.3.5_ana33047_minusn_0003_Crop_series1_cellMask_cropped.tif')
print(movie_mask.shape)
movie_mask = np.where(movie_mask > 0, 1, 0)
for frame in range(len(movie_mask)):
	labeled_mask, num_clusters = label(np.array(movie_mask[frame], dtype=int))
	# plt.imshow(labeled_mask)
	# plt.show()
	# break
	tifffile.imwrite(f'/Users/bebr1814/projects/anabaena/scratch_data/movies/training_images/2020.3.5_ana33047_minusn_0003_Crop_cropped.frame_{frame}.merged_mask.tiff',labeled_mask)

(80, 486, 691)


In [6]:
movie = tifffile.imread('/Users/bebr1814/projects/anabaena/scratch_data/movies/2020.3.5_ana33047_minusn_0003_Crop_cropped.tif')
print(movie.shape) 
for frame in range(movie.shape[0]):
	output_img = movie[frame,:,:,:]
	output_img[0] = normalize(output_img[0])
	output_img[1] = normalize(output_img[1])
	# plt.imshow(output_img[0])
	# plt.show()
	# plt.imshow(output_img[1])
	# plt.show()
	# break
	tifffile.imwrite(f'/Users/bebr1814/projects/anabaena/scratch_data/movies/training_images/2020.3.5_ana33047_minusn_0003_Crop_cropped.frame_{frame}.merged_img.tiff', output_img)

(80, 2, 486, 691)


In [7]:
for f in glob.glob('/Users/bebr1814/projects/anabaena/scratch_data/movies/training_images/2020.3.5_ana33047_minusn_0003_Crop_cropped.frame_*.merged_*.tiff'):
	# get frame number
	frame = int(re.search(r'frame_(\d+)', f).group(1))
	if frame in test_frames:
		shutil.copyfile(f, os.path.join(test_path, os.path.basename(f)))
	else:
		shutil.copyfile(f, os.path.join(train_path, os.path.basename(f)))

# FOV Images

In [8]:
paths = ['/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena','/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena']

In [9]:
def merge_tiffs(brightfield,cy5):
	c1 = normalize(tifffile.imread(brightfield))
	c2 = normalize(tifffile.imread(cy5))
	merged = np.stack([c1,c2],axis=0)
	return merged

In [10]:
for path in paths:
	dirs = glob.glob(os.path.join(path,'*','tif'))
	for d in dirs:
		brightfields = glob.glob(d+'/*.brightfield.tiff')
		cy5s = glob.glob(d+'/*.cy5.tiff')
		brightfields = natsort.natsorted(brightfields)
		cy5 = natsort.natsorted(cy5s)
		
		for b,c in zip(brightfields,cy5):
			if "20241010" in b:
				c1 = normalize(tifffile.imread(b))
				c2 = normalize(tifffile.imread(c))
			elif "20241114" in b:
				c1 = normalize(tifffile.imread(b)[0])
				c2 = normalize(tifffile.imread(c)[0])
			merged = np.stack([c1,c2],axis=0)
			tifffile.imwrite(b.replace('.brightfield.tiff','.merged_img.tiff'),merged)
			# print('saved',b.replace('.brightfield.tiff','.merged.tiff'))


In [11]:
for path in paths:
	for f in glob.glob(os.path.join(path,'*','tif','*_seg.npy')):
		seg = np.load(f,allow_pickle=True).item()
		mask_file = f.replace('_seg.npy','_mask.tiff').replace('brightfield','merged')
		mask = seg['masks']
		# plt.imshow(mask)
		# plt.show()
		# break
		tifffile.imwrite(mask_file,mask)


In [12]:
i = 0
for path in paths:
	for f in glob.glob(os.path.join(path,'*','tif','*.merged_img.tiff')):
		dirname = f.split('/')[-3]
		filename = dirname+'.'+os.path.basename(f)

		# Conditionally add to training data if the mask exists
		if os.path.exists(f.replace('_img','_mask')):

			if i % 10 == 0:
				shutil.copyfile(f,os.path.join(test_path,filename))
				shutil.copyfile(f.replace('_img','_mask'),os.path.join(test_path,filename.replace('merged_img','merged_mask')))
			else:
				shutil.copyfile(f,os.path.join(train_path,filename))
				shutil.copyfile(f.replace('_img','_mask'),os.path.join(train_path,filename.replace('merged_img','merged_mask')))
		
		i += 1
